# Mass-lumping and Local time-stepping

Mass-lumping: Choose a nodal basis, where the interpolation nodes correspond to integration points. Thus, the mass-matrix becomes diagonal.

Well know for first order elements.

More recent for higher order methods:
* G. Cohen, P. Joly, J. E. Roberts, and N. Tordman: Higher order triangular finite elements with mass lumping for the wave equation
, SINUM 38(6), pp 2047-2078 (2000)


* S. Geevers, W.A. Mulder, and J.J.W. van der Vegt:
New higher-order mass-lumped tetrahedral elements for wave propagation modelling: https://arxiv.org/pdf/1803.10065.pdf (2018)


Basic idea for higher order triangles:

$P^2$ triangle with the 6 basis functions does not work with mass-lumping, however adding the interior bubble works ! Then we have 7 integration points on the triangle.


In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import unit_square
from time import sleep, time

maxh = 0.15
mesh = unit_square.GenerateMesh(maxh=maxh)
for l in range(5): 
    mesh.Refine()
    maxh /= 2
mesh = Mesh(mesh)
# Draw (mesh)

In [ ]:
order=2   # order=1 or order=2 is supported

tau = maxh / (5*order)
tend = 2
u0 = exp(-100**2*( (x-0.5)**2 + (y-0.5)**2))
v0 = 0

fes = H1LumpingFESpace(mesh, order=order)  
u,v = fes.TnT()

mform = u*v*dx(intrules=fes.GetIntegrationRules())
aform = grad(u)*grad(v)*dx

m = BilinearForm(mform, diagonal=True).Assemble()
a = BilinearForm(aform).Assemble()
minv = m.mat.Inverse(fes.FreeDofs())  

# or, in short:
# minv = fes.Mass(rho=1).Inverse()

## The Verlet method:

the second-order wave equation 

$$
\frac{\partial^2 u}{\partial t^2} = \Delta u
$$

is discretized with mass matrix $M$ and stiffness matrix $A$ as

$$
M \frac{\partial^2 u}{\partial t^2} = -A u.
$$

The time-derivative is approximated by a second order finite difference stencil:

$$
M \frac{u^{n+1} - 2 u^n + u^{n-1}}{\tau^2} = -A u^n
$$

In [ ]:
gfu = GridFunction(fes)
gfu.Set(u0)

scene = Draw(gfu, order=2, deformation=True, scale=3, euler_angles=[-60,0,-40])
sleep (2)
unew = gfu.vec.CreateVector()
uold = gfu.vec.CreateVector()
uold.data = gfu.vec

with TaskManager(): 
    for n in range(int(tend/tau)):
        unew.data = 2*gfu.vec - uold 
        unew.data -= tau**2 * minv@a.mat * gfu.vec
        uold.data = gfu.vec
        gfu.vec.data = unew.data
        if n % 100 == 0:
            scene.Redraw()

scene.Redraw()

For higher order version and local time-stepping see: [Talk by Marcus Grote](https://team.inria.fr/magique3d/files/2016/03/talk_grote.pdf)

Try examples from there.